In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("student_data.csv")
df.head()

In [ ]:
df.columns = df.columns.str.strip().str.lower()
print(df.columns)

In [ ]:
# Remove unwanted column
df = df.drop(columns=["prediction"], errors="ignore")

# Check missing values
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(5,4))
sns.scatterplot(x=df["study_hours"], y=df["previous_marks"])
plt.title("Study Hours vs Marks")
plt.show()

plt.figure(figsize=(5,4))
sns.boxplot(x=df["final_result"], y=df["attendance"])
plt.title("Attendance vs Result")
plt.show()

In [ ]:
X = df[["study_hours","attendance","previous_marks","assignments","internal_marks"]]
y = df["final_result"]

print("Selected Features:", X.columns.tolist())

In [ ]:
from sklearn.preprocessing import StandardScaler
import pickle

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pickle.dump(scaler, open("scaler.pkl", "wb"))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(),
    "Decision Tree": DecisionTreeClassifier()
}

for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    print(name, "Accuracy:", accuracy_score(y_test, pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

pickle.dump(model, open("model.pkl", "wb"))

print("✅ Model trained and saved")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred = model.predict(X_test)

print(classification_report(y_test, pred))

cm = confusion_matrix(y_test, pred)

sns.heatmap(cm, annot=True, cmap="Purples", fmt="d")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
df["prediction"] = model.predict(X_scaled)

at_risk = df[df["prediction"] == 0]

print("🔴 At-Risk Students:")
print(at_risk)

print("\nTotal At-Risk:", len(at_risk))

In [ ]:
from google.colab import files
files.download("model.pkl")
files.download("scaler.pkl")

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import pickle
import numpy as np

# Load model & scaler
model = pickle.load(open("model.pkl", "rb"))
scaler = pickle.load(open("scaler.pkl", "rb"))

def predict(study_hours, attendance, previous_marks, assignments, internal_marks):
    data = np.array([[study_hours, attendance, previous_marks, assignments, internal_marks]])
    data = scaler.transform(data)

    result = model.predict(data)[0]

    if result == 1:
        return "✅ PASS"
    else:
        return "❌ FAIL"

ui = gr.Interface(
    fn=predict,
    inputs=[
        gr.Slider(0,12,label="Study Hours"),
        gr.Slider(0,100,label="Attendance"),
        gr.Slider(0,100,label="Previous Marks"),
        gr.Slider(0,100,label="Assignments"),
        gr.Slider(0,100,label="Internal Marks")
    ],
    outputs="text",
    title="🎓 Student Performance Predictor"
)

ui.launch()